## 🎯 Objetivo

1. Entender la estructura de **Unity Catalog** (Catálogo → Esquema → Tabla/Volume)
2. Crear tu propio **catálogo, esquema y volume**
3. Cargar datos crudos desde un CSV
4. Realizar un **EDA (Exploratory Data Analysis)** completo
5. Aplicar **CTEs** y **Window Functions** en análisis real
6. Identificar problemas de calidad de datos

---

## 📊 Dataset: Propiedades Inmobiliarias

Datos **reales y crudos** de propiedades inmobiliarias scrapeadas de:
- ZonaProp
- MercadoLibre
- Argenprop

In [0]:
%sql
-- ============================================================
-- PASO 2: Paso 2: Crear los esquemas Landing y Bronze
-- ============================================================

--Usar tu catálogo
USE CATALOG bootcamp;

-- Crear esquema Landing
CREATE SCHEMA IF NOT EXISTS bootcamp.landing
COMMENT 'Esquema para archivos';

--Crear esquema Raw
CREATE SCHEMA IF NOT EXISTS bootcamp.bronze
COMMENT 'Esquema para datos crudos sin procesar';

-- Verificar
SHOW SCHEMAS;

-- ============================================================
-- PASO 3: Crear the Volume para archivos
-- ============================================================
-- Crear volume tipo MANAGED (Databricks administra el storage)
CREATE VOLUME IF NOT EXISTS bootcamp.landing.archivos
COMMENT 'Volume para almacenar archivos CSV crudos';

-- Verificar que se creó
SHOW VOLUMES IN bootcamp.landing;

-- ============================================================
-- PASO 4:  Subir el archivo CSV properties_raw.csv
-- Verificar que el archivo se subió correctamente
-- ============================================================

-- Listar archivos en el volume
LIST '/Volumes/bootcamp/landing/archivos/';


-- ============================================================
-- PASO 5: Crear la tabla properties_bronze
-- ============================================================
-- Primero, eliminamos la tabla si existe (para poder recrearla)
DROP TABLE IF EXISTS bootcamp.bronze.properties_bronze;

-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida

CREATE TABLE bootcamp.bronze.properties_bronze
 SELECT * FROM read_files(
 '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
 format => 'csv',
 header => true
 )
 where url like 'https%' --Filtramos por aquellos links que sean consistentes
 ;

-- Verificar que la tabla se creó correctamente
SHOW TABLES IN bootcamp.bronze;




# EDA Paso 1: Exploración Inicial


In [0]:
--Ejercicio 1.1: Verificar datos disponibles
-- ¿Cuántos registros hay en total en la tabla bootcamp.bronze.properties_bronze?

SELECT count(*) as Total_Registros
from bootcamp.bronze.properties_bronze

In [0]:
 --Explorar estructura 
 --¿Qué columnas tiene la tabla? ¿Qué tipos de datos tienen? 

describe TABLE bootcamp.bronze.properties_bronze

In [0]:
--Ejercicio 1.3: Ver muestra de datos - Muestra las primeras 10 filas del dataset, pero solo las columnas más importantes:
-- id, ubicacion, precio, expensas, tipo_de_operacion, moneda, ambientes, metros_cuadrados_totales,antiguedad, estado, zona. 


SELECT 
    id,
    ubicacion,
    precio,
    expensas,
    tipo_de_operacion,
    moneda,
    ambientes,
    metros_cuadrados_totales,
    antiguedad,
    estado,
    zona
FROM bootcamp.bronze.properties_bronze
limit 10 

# PARTE 2: Análisis de Valores Nulos

Identificar campos vacíos es **crítico** para planificar la limpieza de datos.


In [0]:
--Ejercicio 2.1: Contar nulos por columna 
-- Cuántos valores nulos hay en cada columna importante? 

WITH Total as
(
    SELECT
         count(*) as TotalRegistros
    FROM bootcamp.bronze.properties_bronze
),


RegistrosNulos as 
(
 SELECT
        COUNT(*) - COUNT(id) as nulos_id,
        COUNT(*) - COUNT(ubicacion) as nulos_ubicacion,
        COUNT(*) - COUNT(precio) as nulos_precio,
        COUNT(*) - COUNT(expensas) as nulos_expensas,
        COUNT(*) - COUNT(tipo_de_operacion) as nulos_tipo_de_operacion,
        COUNT(*) - COUNT(moneda) as nulos_moneda,
        COUNT(*) - COUNT(ambientes) as nulos_ambientes,
        COUNT(*) - COUNT(metros_cuadrados_totales) as nulos_metros_cuadrados_totales,
        COUNT(*) - COUNT(antiguedad) as nulos_antiguedad,
        COUNT(*) - COUNT(estado) as nulos_estado,
        COUNT(*) - COUNT(zona)
               
 FROM bootcamp.bronze.properties_bronze
 )

 SELECT 
 T.totalregistros,
 RN.*
 FROM Total T, registrosnulos RN


In [0]:

-- Ejercicio 2.2: Porcentaje de nulos 
--¿Qué porcentaje de registros tienen valores nulos en las columnas clave?

WITH Total as
(
    SELECT
         count(*) as TotalRegistros
    FROM bootcamp.bronze.properties_bronze
),


RegistrosNulos as 
(
 SELECT
        COUNT(*) - COUNT(id) as nulos_id,
        COUNT(*) - COUNT(ubicacion) as nulos_ubicacion,
        COUNT(*) - COUNT(precio) as nulos_precio,
        COUNT(*) - COUNT(expensas) as nulos_expensas,
        COUNT(*) - COUNT(tipo_de_operacion) as nulos_tipo_de_operacion,
        COUNT(*) - COUNT(moneda) as nulos_moneda,
        COUNT(*) - COUNT(ambientes) as nulos_ambientes,
        COUNT(*) - COUNT(metros_cuadrados_totales) as nulos_m2_totales,
        COUNT(*) - COUNT(metros_cuadrados_cubiertos) as nulos_m2_cubiertos,
        COUNT(*) - COUNT(orientacion_cardinal)as nulos_orientacion_cardinal,
        COUNT(*) - COUNT(antiguedad) as nulos_antiguedad,
        COUNT(*) - COUNT(estado) as nulos_estado,
        COUNT(*) - COUNT(zona)
               
 FROM bootcamp.bronze.properties_bronze
 )

 SELECT 
 ROUND((RN.nulos_precio / T.totalregistros) *100,2)  AS pct_NulosPrecio,
 ROUND((RN.nulos_expensas / T.totalregistros) *100,2) AS pct_NulosExpensas,
 ROUND((RN.nulos_ambientes / T.totalregistros) *100,2) AS pct_NulosAmbientes,
 ROUND((RN.nulos_m2_totales / T.totalregistros) *100,2) AS pct_NulosM2Totales,
 ROUND((RN.nulos_m2_cubiertos/ T.totalregistros) *100,2) AS pct_NulosM2Totales,
 ROUND((RN.nulos_orientacion_cardinal / T.totalregistros) *100,2) AS pct_NulosOrientacionCardinal,
 ROUND((RN.nulos_antiguedad / T.totalregistros) *100,2) AS PorcentajeNulosAntiguedad
  
 FROM Total T, registrosnulos RN

In [0]:
-- ============================================================
-- EDA 2.3: Identificar columnas con >50% de valores nulos
-- ============================================================
-- Columnas con >50% nulos son críticas: hay que decidir si eliminarlas o imputarlas

WITH totales AS (
    SELECT COUNT(*) as total FROM bootcamp.bronze.properties_bronze
),

pct_nulos AS (
    SELECT 
        ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as precio,
        ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as expensas,
        ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as ambientes,
        ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as metros_cuadrados_totales,
        ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as metros_cuadrados_cubiertos,
        ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as orientacion_cardinal,
        ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as antiguedad,
        ROUND((t.total - COUNT(piso)) * 100.0 / t.total, 2) as piso,
        ROUND((t.total - COUNT(cochera)) * 100.0 / t.total, 2) as cochera
    FROM bootcamp.bronze.properties_bronze
    CROSS JOIN totales t
    GROUP BY t.total
)

SELECT 

    NombreColumna,
    porcentaje_nulos
    
FROM pct_nulos
UNPIVOT (
    porcentaje_nulos FOR NombreColumna IN(
        Precio, Expensas, Ambientes, metros_Cuadrados_Totales, metros_Cuadrados_Cubiertos, Orientacion_Cardinal, Antiguedad, Piso, Cochera)
        )
WHERE porcentaje_nulos > 50

# PARTE 3: Cardinalidad y Distribución

In [0]:


--Distribución de tipo de operación
-- ¿Qué valores únicos hay en tipo_de_operacion? ¿Cuántos registros hay de cada tipo? ¿Qué porcentaje representa cada uno?

SELECT
    DISTINCT (tipo_de_operacion), 
    COUNT(*) AS TotalRegistros,
    ROUND(COUNT(*) * 100 / SUM(COUNT(*))  OVER(),2) AS pct

FROM bootcamp.bronze.properties_bronze
GROUP by tipo_de_operacion
ORDER BY totalregistros DESC


In [0]:
--Ejercicio 3.2: Distribución de moneda
-- ¿Qué monedas hay en el dataset? ¿Cuántos registros hay de cada una? ¿Hay alguna moneda que no esperabas? 

SELECT
    moneda, 
    COUNT(*) AS TotalRegistros,
   ROUND(count(*)/SUM(COUNT(*)) OVER() * 100 ,2) AS pct 


FROM bootcamp.bronze.properties_bronze
GROUP by moneda
ORDER BY totalregistros DESC

In [0]:
--Ejercicio 3.3: Distribución de ambientes 
-- ¿Cuántas propiedades hay por cantidad de ambientes? ¿Cuál es la distribución?

SELECT
    ambientes, 
    COUNT(*) AS TotalRegistros,
    ROUND(COUNT(*) *100 / SUM((COUNT(*))) OVER(),2) AS  pct

FROM bootcamp.bronze.properties_bronze
GROUP BY ambientes
ORDER BY ambientes ASC

In [0]:
%sql
-- ============================================================
-- EDA 3.4: Top 15 zonas con más propiedades
-- ============================================================

SELECT
    zona, 
    count(*) as totalPropiedades,
    ROUND(count(*)*100/ sum(count(*)) over (),2) as pct
FROM bootcamp.bronze.properties_bronze 
GROUP by zona 
ORDER BY totalpropiedades desc
limit 15

In [0]:
-- ============================================================
-- EDA 3.5: Distribución de estado de las propiedades
-- ============================================================

SELECT 
    estado,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM bootcamp.bronze.properties_bronze
GROUP BY estado
ORDER BY cantidad DESC;

## EDA Paso 4: Estadísticas Descriptivas de Variables Numéricas

Analizar rangos, promedios y detectar outliers en las columnas numéricas.

In [0]:
-- Esta falla intencionalmente para ver que no podemos hacer cálculos porque los formatos de columas no son los adecuados
SELECT
   
    moneda,
    MAX(precio),
    MIN(precio)

FROM bootcamp.bronze.properties_bronze
WHERE precio > 0
GROUP BY  moneda
ORDER BY   moneda ASC


In [0]:
-- Tabla temporal para cambiar los formatos de las columnas y poder  hacer cálculos. 

CREATE OR REPLACE TEMPORARY VIEW propiedades_clean AS

SELECT 
    TRY_CAST(precio AS DOUBLE) AS precio,
    TRY_CAST(ambientes AS DOUBLE) AS ambientes,
    TRY_CAST(metros_cuadrados_totales AS DOUBLE) AS metros_cuadrados_totales,
    TRY_CAST(metros_cuadrados_cubiertos AS DOUBLE) AS metros_cuadrados_cubiertos,
    TRY_CAST(antiguedad AS DOUBLE) AS antiguedad,
    moneda,
    tipo_de_operacion,
    id,
    ubicacion,
    numero,
    calle,
    expensas,
    orientacion_cardinal,
    orientacion_inmueble,
    piso,
    cochera,
    estado,
    tipo_vendedor,
    url,
    zona,
    fecha,
    hora

FROM bootcamp.bronze.properties_bronze;

In [0]:
-- ============================================================
-- EDA 4.1: Estadísticas de PRECIO (separado por moneda y tipo de operación)
-- ============================================================

SELECT 
    moneda,
    tipo_de_operacion,
    COUNT(*) as cantidad,
    ROUND(MIN(precio), 2) as precio_min,
    ROUND(MAX(precio), 2) as precio_max,
    ROUND(AVG(precio), 2) as precio_promedio,
    ROUND(PERCENTILE(precio, 0.5), 2) as precio_mediana,
    ROUND(PERCENTILE(precio, 0.25), 2) as precio_p25,
    ROUND(PERCENTILE(precio, 0.75), 2) as precio_p75

FROM propiedades_clean
WHERE precio IS NOT NULL AND precio > 0
GROUP BY moneda, tipo_de_operacion
ORDER BY cantidad DESC;


In [0]:
-- ============================================================
-- EDA 4.1b (BONUS — no está en el PDF): Estadísticas de PRECIO (filtrado)
-- Solo USD/ARS, operaciones principales, sin outliers (P1-P99)
-- ============================================================
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT 
    p.moneda,
    p.tipo_de_operacion,
    COUNT(*) AS cantidad,
    ROUND(MIN(p.precio), 2) AS precio_min,
    ROUND(MAX(p.precio), 2) AS precio_max,
    ROUND(AVG(p.precio), 2) AS precio_promedio,
    ROUND(PERCENTILE(p.precio, 0.5), 2) AS precio_mediana,
    ROUND(PERCENTILE(p.precio, 0.25), 2) AS precio_p25,
    ROUND(PERCENTILE(p.precio, 0.75), 2) AS precio_p75
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
  GROUP BY p.moneda, p.tipo_de_operacion
  ORDER BY cantidad DESC

In [0]:
--vista temporal 2- valores formateados, extermos retirados y solo monedas de interés. 


CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS

WITH limites as (
    SELECT
        moneda, 
        tipo_de_operacion,
        percentile(precio,0.01) as P01,
        percentile(precio,0.99) as P99
    FROM propiedades_clean
    WHERE precio > 0
    AND moneda IN ( 'USD', 'ARS')
    AND tipo_de_operacion  IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
)

SELECT 
    p.*
FROM propiedades_clean p
INNER JOIN limites l
    ON p.moneda = l.moneda
    AND p.tipo_de_operacion = l.tipo_de_operacion
WHERE p.precio BETWEEN l.P01 AND l.P99



In [0]:
--Ejercicio 4.2: Estadísticas de metros cuadrados

SELECT
    'Metros Cuadrados totales' as TipoMetros,
    MAX(metros_cuadrados_totales) as max_metros_totales,
    MIN(metros_cuadrados_totales) AS min_metros_totales,
    AVG(metros_cuadrados_totales) AS avg_metros_totales,
    percentile(metros_cuadrados_totales, 0.5) AS mediana_metros_totales
    
FROM propiedades_clean_2 
where metros_cuadrados_totales > 0


UNION ALL

SELECT
    'Metros Cuadrados cubiertos' as TipoMetros,
    MAX(metros_cuadrados_cubiertos) as max_metros_cubiertos,
    MIN(metros_cuadrados_cubiertos) AS min_metros_cubiertos,
    AVG(metros_cuadrados_cubiertos) AS avg_metros_cubiertos,
    percentile(metros_cuadrados_cubiertos, 0.5) AS mediana_metros_cubiertos
    
FROM propiedades_clean_2 
where metros_cuadrados_cubiertos > 0

In [0]:

--Ejercicio 4.3: Análisis de antigüedad Objetivo: Identificar valores placeholder en antigüedad.

SELECT
    Antiguedad, 
    COUNT(*) AS cantidad

FROM propiedades_clean_2
GROUP BY Antiguedad
ORDER BY cantidad DESC

In [0]:

--Ejercicio 4.4: Estadísticas de  antigüedad sin placeholders Objetivo: Calcular estadísticas reales excluyendo valores placeholder

SELECT
   
    COUNT(*) AS cantidad,
    MAX(Antiguedad) as max_antiguedad,
    MIN(Antiguedad) as min_antiguedad,
    AVG(Antiguedad) AS avg_antiguedad,
    PERCENTILE(Antiguedad, 0.5) as mediana_antiguedad

FROM propiedades_clean_2
WHERE 
    Antiguedad NOT IN (999)
    AND Antiguedad IS NOT NULL
    AND Antiguedad >= 0
    
    ORDER BY cantidad DESC

# PARTE 5: Detección de Problemas de Calidad

In [0]:

WITH TotalRegistros as (

SELECT 
    COUNT (*) AS cantidad
    FROM propiedades_clean_2
),

Preciosproblematicoas as (
SELECT 
        Count(*) as cantidad
FROM propiedades_clean_2
WHERE precio <=0  OR precio is NULL), 


Metrosproblematicoas as (
SELECT 
        Count(*) as cantidad
FROM propiedades_clean_2
WHERE metros_cuadrados_totales <=0  OR metros_cuadrados_totales is NULL), 

Antiguedadproblematicas as (
SELECT 
        Count(*) as cantidad
FROM propiedades_clean_2
WHERE Antiguedad IN (999) )



SELECT 
'Precios Problematicos' as Problema,
PP.cantidad,
TR.cantidad- PP.cantidad as RegistrosValidos
FROM Preciosproblematicoas PP, TotalRegistros TR

union ALL

SELECT 
'Metros Problematicos' as Problema,
MR.cantidad, 
TR.cantidad-MR.cantidad as RegistrosValidos
FROM Metrosproblematicoas MR, TotalRegistros TR

union ALL

SELECT 
'Antiguedad Problematicos' as Problema,
AP.cantidad,
TR.cantidad-AP.cantidad as RegistrosValidos
FROM Antiguedadproblematicas AP, TotalRegistros TR


In [0]:
--Ejercicio 5.2 Detectar duplicados


WITH repetidos AS (
SELECT
    precio, 
    URL, 
    count(*) as totalRegistros,
    count(*) -1 as duplicados

FROM propiedades_clean_2
GROUP BY 1,2
having count(*) > 1
ORDER BY TotalRegistros DESC )

SELECT
    COUNT(*) propiedadesDuplicadas, 
    sum(totalRegistros) as totalDuplicados,
    Sum(duplicados) as RegistrosExtra
  FROM repetidos  



In [0]:
--Ejercicio 5.3 Ver ejemplos de duplicado

SELECT
    precio, 
    URL, 
    count(*) as totalRegistros,
    count(*) -1 as duplicados

FROM propiedades_clean_2
GROUP BY 1,2
having count(*) > 1
ORDER BY TotalRegistros DESC 
LIMIT 10

In [0]:
--Ejercicio 5.4 Detectar outliers en precio

WITH limites as (
SELECT
Moneda, 
percentile(precio, 0.01) as P1,
percentile(precio, 0.99) as P99
FROM propiedades_clean_2
GROUP BY  moneda),


clasificacion as (
SELECT
p.Moneda,
p.precio,
CASE WHEN p.precio < l.P1 THEN 'Outlier Bajo'
WHEN p.precio > l.P99 THEN 'Outlier Alto'
ELSE 'Normal' END as Outliers
FROM propiedades_clean_2 p
JOIN limites l ON p.moneda = l.moneda)

SELECT 
    Outliers,
    count(*) as Cantidad

FROM clasificacion 
GROUP BY Outliers
ORDER BY Cantidad DESC



# PARTE 6-7: Análisis Avanzado y Documentación

In [0]:

--Ejercicio 6.1: Ranking de zonas por precio Moneda ARS y operación Alquiler

SELECT

    zona, 
    COUNT(*) as cantidadPropiedades,    
    ROUND(AVG(precio),2) as  precioPromedio,
    ROW_NUMBER() OVER ( ORDER BY AVG(precio) DESC) AS Rank

FROM propiedades_clean_2
WHERE 
precio >0 OR  precio IS NOT null
AND moneda = 'ARS'
AND tipo_de_operacion = 'Alquiler'
GROUP by zona

In [0]:
SELECT

    COUNT(*) as cantidadPropiedades,    
    AVG(precio) precioPromedioGeneral
    
FROM propiedades_clean_2
WHERE 
precio >0 OR  precio IS NOT null
AND moneda = 'ARS'
AND tipo_de_operacion = 'Alquiler'

In [0]:

--Ejercicio 6.2: Comparar con promedio general - Sin cross join 

WITH zonas as (

    SELECT
        zona, 
        COUNT(*) as cantidadPropiedades,    
        AVG(precio) promedioZona

        FROM propiedades_clean_2

    WHERE 
        precio >0 OR  precio IS NOT null
        AND moneda = 'ARS'
        AND tipo_de_operacion = 'Alquiler'
    GROUP BY zona )

SELECT
zona,
cantidadPropiedades,  
promedioZona,  
AVG(promedioZona) over() as promedioGeneral, 
promedioZona- AVG(promedioZona) over() as diferencia

FROM zonas


In [0]:
--Ejercicio 6.2: Comparar con promedio general - Con cross join 

WITH zonas as (

    SELECT
        zona, 
        COUNT(*) as cantidadPropiedades,    
        AVG(precio) promedioZona

        FROM propiedades_clean_2

    WHERE 
        precio >0 OR  precio IS NOT null
        AND moneda = 'ARS'
        AND tipo_de_operacion = 'Alquiler'
    GROUP BY zona ),


General as (

    SELECT
        AVG(precio) as promedioGeneral
    FROM propiedades_clean_2
     WHERE 
        precio >0 OR  precio IS NOT null
        AND moneda = 'ARS'
        AND tipo_de_operacion = 'Alquiler'

)


SELECT
z.*,
g.promedioGeneral

FROM zonas z, General g
